In [1]:
# ============================================
# 03_retrieval_system.ipynb
# Purpose: Implement Hybrid Search and Re-ranking
# ============================================

# Cell 1: Import libraries
import json
import pinecone
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
import numpy as np
from typing import List, Dict, Tuple
import re
from collections import defaultdict
import time
from dotenv import load_dotenv
import os

load_dotenv()

print("Libraries loaded!")

d:\IBA\Sem-6\NLP with Deep Learning\Assignment 3\pakistan_history-rag\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries loaded!


In [2]:
# ============================================
# Cell 2: Initialize components
# ============================================

# Initialize embedding model (same as before)
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Initialize cross-encoder for re-ranking
# This is the key for the "Re-ranking" requirement
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

# Connect to Pinecone
from pinecone import Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
index = pc.Index("pakistan-history-rag")

print("✅ All models and indexes initialized!")
print(f"Embedding dimension: {embedding_model.get_sentence_embedding_dimension()}")
print(f"Cross-encoder model loaded")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5155.66it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 6000.03it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ All models and indexes initialized!
Embedding dimension: 384
Cross-encoder model loaded


In [3]:
# ============================================
# Cell 3: Implement BM25 (Keyword Search)
# ============================================

class BM25Retriever:
    """
    BM25 retriever for keyword-based search
    """
    def __init__(self, chunks):
        """
        chunks: List of chunk dictionaries with 'text' and 'id'
        """
        self.chunks = chunks
        # Tokenize all chunks
        tokenized_chunks = [self._tokenize(chunk['text']) for chunk in chunks]
        self.bm25 = BM25Okapi(tokenized_chunks)
    
    def _tokenize(self, text):
        """Simple tokenizer - lowercase and split on non-alphanumeric"""
        return re.findall(r'\w+', text.lower())
    
    def search(self, query, top_k=10):
        """Search with BM25"""
        tokenized_query = self._tokenize(query)
        scores = self.bm25.get_scores(tokenized_query)
        
        # Get top-k indices
        top_indices = np.argsort(scores)[::-1][:top_k]
        
        results = []
        for idx in top_indices:
            if scores[idx] > 0:  # Only return results with positive scores
                results.append({
                    'id': self.chunks[idx]['id'],
                    'text': self.chunks[idx]['text'],
                    'score': float(scores[idx]),
                    'metadata': self.chunks[idx].get('metadata', {})
                })
        
        return results

# Load all chunks from Pinecone to build BM25 index
# (We only need to do this once, but for a real system we'd optimize)

def get_all_chunks_from_pinecone(index, limit=10000):
    """
    Retrieve all chunks from Pinecone (for BM25 indexing)
    Note: In production, you'd want a more efficient approach
    """
    # This is a simplified approach - for large datasets, you'd paginate
    # Since we have <1000 chunks, this is fine
    stats = index.describe_index_stats()
    total_vectors = stats['total_vector_count']
    
    # Pinecone doesn't have a "get all" method, so we'll query with dummy vector
    # This is a hack - for real production, store chunks separately
    print("Retrieving all chunks from Pinecone...")
    
    # We'll use a random vector to get many results
    random_vector = np.random.rand(384).tolist()
    results = index.query(
        vector=random_vector,
        top_k=min(total_vectors, 10000),
        include_metadata=True
    )
    
    chunks = []
    for match in results['matches']:
        chunks.append({
            'id': match['id'],
            'text': match['metadata']['text'],
            'metadata': match['metadata']
        })
    
    print(f"Retrieved {len(chunks)} chunks")
    return chunks

# Get all chunks
all_chunks = get_all_chunks_from_pinecone(index)
bm25_retriever = BM25Retriever(all_chunks)
print("✅ BM25 index built!")


Retrieving all chunks from Pinecone...
Retrieved 10000 chunks
✅ BM25 index built!


In [4]:
# ============================================
# Cell 4: Implement Hybrid Search with RRF (COMPLETE FIXED VERSION)
# ============================================

def reciprocal_rank_fusion(semantic_results, bm25_results, k=60):
    """
    Combine search results using Reciprocal Rank Fusion (RRF)
    """
    from collections import defaultdict
    
    fusion_scores = defaultdict(float)
    result_data = {}
    
    # Add semantic results
    for rank, result in enumerate(semantic_results, start=1):
        result_id = result['id']
        fusion_scores[result_id] += 1 / (k + rank)
        result_data[result_id] = result
    
    # Add BM25 results
    for rank, result in enumerate(bm25_results, start=1):
        result_id = result['id']
        fusion_scores[result_id] += 1 / (k + rank)
        if result_id not in result_data:
            result_data[result_id] = result
    
    # Sort by fusion score
    sorted_results = sorted(fusion_scores.items(), key=lambda x: x[1], reverse=True)
    
    # Build final results list
    final_results = []
    for result_id, score in sorted_results:
        result = result_data.get(result_id, {})
        result['fusion_score'] = score
        final_results.append(result)
    
    return final_results

def hybrid_search(query, index, embedding_model, bm25_retriever, top_k=10):
    """
    Perform hybrid search: Semantic + BM25 with RRF fusion
    """
    # Step 1: Semantic search with Pinecone
    query_embedding = embedding_model.encode(query)
    semantic_results_raw = index.query(
        vector=query_embedding.tolist(),
        top_k=top_k*2,
        include_metadata=True
    )
    
    # Format semantic results
    semantic_results = []
    for match in semantic_results_raw['matches']:
        semantic_results.append({
            'id': match['id'],
            'text': match['metadata'].get('text', ''),
            'score': match['score'],
            'metadata': match['metadata']
        })
    
    # Step 2: BM25 search
    bm25_results = bm25_retriever.search(query, top_k=top_k*2)
    
    # Step 3: Fuse with RRF
    fused_results = reciprocal_rank_fusion(semantic_results, bm25_results)
    
    # Ensure all results have 'text' field
    for result in fused_results:
        if 'text' not in result and 'metadata' in result and 'text' in result['metadata']:
            result['text'] = result['metadata']['text']
    
    return fused_results[:top_k]

def rerank_results(query, results, cross_encoder, top_k=5):
    """
    Re-rank search results using a cross-encoder model
    """
    if not results:
        return results
    
    # Helper to get text
    def get_text(r):
        if 'text' in r:
            return r['text']
        elif 'metadata' in r and 'text' in r['metadata']:
            return r['metadata']['text']
        else:
            return ""
    
    # Prepare pairs for cross-encoder
    pairs = [(query, get_text(result)) for result in results]
    
    # Get relevance scores
    scores = cross_encoder.predict(pairs)
    
    # Add scores to results
    for i, result in enumerate(results):
        result['rerank_score'] = float(scores[i])
    
    # Sort by rerank score
    reranked = sorted(results, key=lambda x: x['rerank_score'], reverse=True)
    
    return reranked[:top_k]

def complete_retrieval_pipeline(query, index, embedding_model, bm25_retriever, 
                                cross_encoder, use_rerank=True, final_k=5):
    """
    Complete retrieval pipeline
    """
    # Step 1: Hybrid search
    hybrid_results = hybrid_search(query, index, embedding_model, bm25_retriever, top_k=20)
    
    # Step 2: Re-rank if requested
    if use_rerank:
        final_results = rerank_results(query, hybrid_results, cross_encoder, top_k=final_k)
    else:
        final_results = hybrid_results[:final_k]
    
    return final_results

print("✅ Hybrid search and re-ranking functions defined successfully!")

✅ Hybrid search and re-ranking functions defined successfully!


In [5]:
# ============================================
# Cell 5: Implement Re-ranking with Cross-Encoder
# ============================================

def rerank_results(query, results, cross_encoder, top_k=5):
    """
    Re-rank search results using a cross-encoder model
    This is more accurate than bi-encoder (semantic search) because
    it considers the interaction between query and document
    """
    if not results:
        return results
    
    # Prepare pairs for cross-encoder
    pairs = [(query, result['text']) for result in results]
    
    # Get relevance scores
    scores = cross_encoder.predict(pairs)
    
    # Add scores to results
    for i, result in enumerate(results):
        result['rerank_score'] = float(scores[i])
    
    # Sort by rerank score
    reranked = sorted(results, key=lambda x: x['rerank_score'], reverse=True)
    
    return reranked[:top_k]

def complete_retrieval_pipeline(query, index, embedding_model, bm25_retriever, 
                                cross_encoder, use_rerank=True, final_k=5):
    """
    Complete retrieval pipeline:
    1. Hybrid search (Semantic + BM25 + RRF)
    2. Optional: Re-ranking with cross-encoder
    3. Return top-k results
    """
    # Step 1: Hybrid search to get initial candidates
    hybrid_results = hybrid_search(query, index, embedding_model, bm25_retriever, top_k=20)
    
    # Step 2: Re-rank if requested
    if use_rerank:
        final_results = rerank_results(query, hybrid_results, cross_encoder, top_k=final_k)
    else:
        final_results = hybrid_results[:final_k]
    
    return final_results

In [6]:
# ============================================
# Cell 6: Compare retrieval methods (FIXED VERSION)
# ============================================

def compare_retrieval_methods(query, index, embedding_model, bm25_retriever, cross_encoder):
    """
    Compare different retrieval strategies for the ablation study
    """
    print(f"\n🔍 Query: {query}")
    print("="*60)
    
    # Helper function to safely get text from result
    def get_text_from_result(result):
        """Extract text from result whether it's in 'text' or 'metadata'"""
        if 'text' in result:
            return result['text']
        elif 'metadata' in result and 'text' in result['metadata']:
            return result['metadata']['text']
        else:
            return "[No text available]"
    
    # Method 1: Semantic only
    start = time.time()
    query_embedding = embedding_model.encode(query)
    semantic_results = index.query(
        vector=query_embedding.tolist(),
        top_k=5,
        include_metadata=True
    )
    semantic_time = time.time() - start
    
    print(f"\n📊 Method 1: Semantic Only (Time: {semantic_time:.3f}s)")
    for i, match in enumerate(semantic_results['matches'], 1):
        # Safely get source title
        source_title = match.get('metadata', {}).get('source_title', 'Unknown')
        # Safely get text
        text = match.get('metadata', {}).get('text', '[No text]')
        print(f"  {i}. Score: {match['score']:.3f} - {source_title}")
        print(f"     {text[:100]}...")
    
    # Method 2: Hybrid without reranking
    start = time.time()
    hybrid_no_rerank = hybrid_search(query, index, embedding_model, bm25_retriever, top_k=5)
    hybrid_time = time.time() - start
    
    print(f"\n📊 Method 2: Hybrid + RRF (No Re-rank) (Time: {hybrid_time:.3f}s)")
    for i, result in enumerate(hybrid_no_rerank, 1):
        fusion_score = result.get('fusion_score', 0)
        text = get_text_from_result(result)
        print(f"  {i}. Fusion Score: {fusion_score:.3f}")
        print(f"     {text[:100]}...")
    
    # Method 3: Hybrid + Re-ranking
    start = time.time()
    hybrid_with_rerank = complete_retrieval_pipeline(
        query, index, embedding_model, bm25_retriever, cross_encoder, 
        use_rerank=True, final_k=5
    )
    rerank_time = time.time() - start
    
    print(f"\n📊 Method 3: Hybrid + RRF + Re-ranking (Time: {rerank_time:.3f}s)")
    for i, result in enumerate(hybrid_with_rerank, 1):
        rerank_score = result.get('rerank_score', 0)
        text = get_text_from_result(result)
        print(f"  {i}. Rerank Score: {rerank_score:.3f}")
        print(f"     {text[:100]}...")
    
    return {
        'semantic': semantic_results['matches'],
        'hybrid_no_rerank': hybrid_no_rerank,
        'hybrid_rerank': hybrid_with_rerank,
        'times': {
            'semantic': semantic_time,
            'hybrid': hybrid_time,
            'rerank': rerank_time
        }
    }

# Also need to fix the rerank_results and complete_retrieval_pipeline functions
# Run this cell FIRST to redefine them properly

def rerank_results(query, results, cross_encoder, top_k=5):
    """
    Re-rank search results using a cross-encoder model
    FIXED to handle results with metadata
    """
    if not results:
        return results
    
    # Helper to get text
    def get_text(r):
        if 'text' in r:
            return r['text']
        elif 'metadata' in r and 'text' in r['metadata']:
            return r['metadata']['text']
        else:
            return ""
    
    # Prepare pairs for cross-encoder
    pairs = [(query, get_text(result)) for result in results]
    
    # Get relevance scores
    scores = cross_encoder.predict(pairs)
    
    # Add scores to results
    for i, result in enumerate(results):
        result['rerank_score'] = float(scores[i])
    
    # Sort by rerank score
    reranked = sorted(results, key=lambda x: x['rerank_score'], reverse=True)
    
    return reranked[:top_k]

def complete_retrieval_pipeline(query, index, embedding_model, bm25_retriever, 
                                cross_encoder, use_rerank=True, final_k=5):
    """
    Complete retrieval pipeline:
    1. Hybrid search (Semantic + BM25 + RRF)
    2. Optional: Re-ranking with cross-encoder
    3. Return top-k results
    """
    # Step 1: Hybrid search to get initial candidates
    hybrid_results = hybrid_search(query, index, embedding_model, bm25_retriever, top_k=20)
    
    # Step 2: Re-rank if requested
    if use_rerank:
        final_results = rerank_results(query, hybrid_results, cross_encoder, top_k=final_k)
    else:
        final_results = hybrid_results[:final_k]
    
    return final_results

# Now run the comparison
print("\n" + "="*60)
print("COMPARING RETRIEVAL METHODS")
print("="*60)

test_queries = [
    "Who was the founder of Pakistan?",
    "What happened during the 1971 war?",
    "When was the Lahore Resolution passed?"
]

comparison_results = {}
for query in test_queries:
    try:
        comparison_results[query] = compare_retrieval_methods(
            query, index, embedding_model, bm25_retriever, cross_encoder
        )
    except Exception as e:
        print(f"Error processing query '{query}': {e}")
        import traceback
        traceback.print_exc()

print("\n✅ Comparison complete!")


COMPARING RETRIEVAL METHODS

🔍 Query: Who was the founder of Pakistan?

📊 Method 1: Semantic Only (Time: 0.364s)
  1. Score: 0.701 - Muhammad Ali Jinnah
     Muhammad Ali Jinnah (born Mahomedali Jinnahbhai; 25 December 1876 – 11 September 1948) was a barrist...
  2. Score: 0.695 - Chaudhry Rehmat Ali
     Choudhry Rahmat Ali (16 November 1897 – 3 February 1951) was a Muslim nationalist activist who is cr...
  3. Score: 0.695 - Constitution of Pakistan of 1956
     . I would like to remind the house that the Father of the Nation, Quaid-i-Azam, gave expression of h...
  4. Score: 0.692 - Liaquat Ali Khan
     and the Indian National Congress had both accepted the idea of Pakistan, which came into existence o...
  5. Score: 0.692 - Islamization in Pakistan
     == Background and history ==

Pakistan was founded on the basis of securing a sovereign homeland for...

📊 Method 2: Hybrid + RRF (No Re-rank) (Time: 0.242s)
  1. Fusion Score: 0.016
     Muhammad Ali Jinnah (born Mahomedali Jinna

In [7]:
# ============================================
# Cell 7: Save the retrieval functions for the app
# ============================================

# We'll save the BM25 index data and configuration
import pickle

# Save BM25 index and chunks for the app
with open('bm25_index.pkl', 'wb') as f:
    pickle.dump({
        'bm25': bm25_retriever.bm25,
        'chunks': all_chunks,
        'tokenizer': bm25_retriever._tokenize
    }, f)

print("✅ Saved BM25 index for the Streamlit app")

# Also save the model names
with open('model_config.json', 'w') as f:
    json.dump({
        'embedding_model': 'sentence-transformers/all-MiniLM-L6-v2',
        'cross_encoder': 'cross-encoder/ms-marco-MiniLM-L-6-v2',
        'index_name': 'pakistan-history-rag'
    }, f)

print("✅ Phase 3 complete! Retrieval system implemented and tested.")

✅ Saved BM25 index for the Streamlit app
✅ Phase 3 complete! Retrieval system implemented and tested.


In [8]:
# ============================================
# Cell 8: Latency Benchmark
# Measures per-stage and end-to-end response time
# across multiple queries and all three pipeline configs.
# Results are saved to latency_results.csv for the report.
# ============================================

import time
import numpy as np
import pandas as pd

benchmark_queries = [
    "When did Pakistan gain independence from British rule?",
    "Who was the first Governor-General of Pakistan?",
    "What was the Objective Resolution of 1949?",
    "What were the main causes of the Indo-Pakistani War of 1965?",
    "How did Bangladesh separate from Pakistan in 1971?",
    "What is the significance of the Indus Valley Civilization?",
    "Which ancient empires ruled the region that is now Pakistan?",
    "What reforms did Zulfikar Ali Bhutto implement?",
    "What was the Kargil War and when did it occur?",
    "Describe the role of the All India Muslim League.",
]

N_RUNS = 3   # average over multiple runs to reduce noise

records = []

print("=" * 65)
print("LATENCY BENCHMARK")
print("=" * 65)
print(f"Queries: {len(benchmark_queries)} | Runs per query: {N_RUNS}")
print()

for q_idx, query in enumerate(benchmark_queries):
    row = {"query": query}

    # ── Stage 1: Query embedding ──────────────────────────────────────────
    emb_times = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        query_embedding = embedding_model.encode(query)
        emb_times.append(time.perf_counter() - t0)
    row["embedding_ms"] = round(np.mean(emb_times) * 1000, 1)

    # ── Stage 2a: Pinecone semantic retrieval ─────────────────────────────
    pine_times = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        _ = index.query(vector=query_embedding.tolist(), top_k=20,
                        include_metadata=True)
        pine_times.append(time.perf_counter() - t0)
    row["pinecone_ms"] = round(np.mean(pine_times) * 1000, 1)

    # ── Stage 2b: BM25 retrieval ─────────────────────────────────────────
    bm25_times = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        _ = bm25_retriever.search(query, top_k=20)
        bm25_times.append(time.perf_counter() - t0)
    row["bm25_ms"] = round(np.mean(bm25_times) * 1000, 1)

    # ── Stage 3: Hybrid search (Pinecone + BM25 + RRF) ───────────────────
    hybrid_times = []
    for _ in range(N_RUNS):
        t0 = time.perf_counter()
        hybrid_results = hybrid_search(query, index, embedding_model,
                                       bm25_retriever, top_k=20)
        hybrid_times.append(time.perf_counter() - t0)
    row["hybrid_retrieval_ms"] = round(np.mean(hybrid_times) * 1000, 1)

    # ── Stage 4: Cross-encoder re-ranking (top-20 → top-5) ───────────────
    # Use results from last hybrid run
    rerank_times = []
    for _ in range(N_RUNS):
        candidates = hybrid_search(query, index, embedding_model,
                                   bm25_retriever, top_k=20)
        t0 = time.perf_counter()
        _ = rerank_results(query, candidates, cross_encoder, top_k=5)
        rerank_times.append(time.perf_counter() - t0)
    row["reranking_ms"] = round(np.mean(rerank_times) * 1000, 1)

    # ── End-to-end: Hybrid + Re-rank (excluding LLM generation) ──────────
    row["e2e_retrieval_ms"] = round(
        row["hybrid_retrieval_ms"] + row["reranking_ms"], 1
    )

    records.append(row)
    print(f"[{q_idx+1:2d}/{len(benchmark_queries)}] {query[:55]:<55}")
    print(f"       embed={row['embedding_ms']:.0f}ms  "
          f"pinecone={row['pinecone_ms']:.0f}ms  "
          f"bm25={row['bm25_ms']:.0f}ms  "
          f"rerank={row['reranking_ms']:.0f}ms  "
          f"e2e={row['e2e_retrieval_ms']:.0f}ms")

# ── Summary ──────────────────────────────────────────────────────────────
df = pd.DataFrame(records)
df.to_csv("latency_results.csv", index=False)

print()
print("=" * 65)
print("SUMMARY (mean across all queries)")
print("=" * 65)
cols = ["embedding_ms", "pinecone_ms", "bm25_ms",
        "hybrid_retrieval_ms", "reranking_ms", "e2e_retrieval_ms"]
for col in cols:
    print(f"  {col:<25}: {df[col].mean():.1f} ms  "
          f"(min {df[col].min():.1f} / max {df[col].max():.1f})")

print()
print("✅ Saved latency_results.csv")


LATENCY BENCHMARK
Queries: 10 | Runs per query: 3

[ 1/10] When did Pakistan gain independence from British rule? 
       embed=16ms  pinecone=220ms  bm25=14ms  rerank=250ms  e2e=495ms
[ 2/10] Who was the first Governor-General of Pakistan?        
       embed=8ms  pinecone=222ms  bm25=16ms  rerank=251ms  e2e=502ms
[ 3/10] What was the Objective Resolution of 1949?             
       embed=15ms  pinecone=222ms  bm25=15ms  rerank=218ms  e2e=469ms
[ 4/10] What were the main causes of the Indo-Pakistani War of 
       embed=10ms  pinecone=226ms  bm25=24ms  rerank=226ms  e2e=483ms
[ 5/10] How did Bangladesh separate from Pakistan in 1971?     
       embed=9ms  pinecone=228ms  bm25=15ms  rerank=269ms  e2e=516ms
[ 6/10] What is the significance of the Indus Valley Civilizati
       embed=8ms  pinecone=227ms  bm25=16ms  rerank=256ms  e2e=513ms
[ 7/10] Which ancient empires ruled the region that is now Paki
       embed=10ms  pinecone=222ms  bm25=19ms  rerank=276ms  e2e=527ms
[ 8/10] What r